# Preprocessing a subset of the realistic Dataset
Especially labeling and checking which trajectories should be used will be done here

In [24]:
import numpy as np
import pandas as pd
import sys
import shutil
import os

In [25]:
def angle_to_axis(vx, vy, vz, zhat):
    axis = np.array((vx, vy, vz))
    axis_norm = axis / np.linalg.norm(axis)
    reference_norm = np.array(zhat) / np.linalg.norm(np.array(zhat))
    cos_angle = np.dot(axis_norm, reference_norm)
    cos_theta = np.clip(cos_angle, -1.0, 1.0)
    angle_degrees = np.degrees(np.arccos(abs(cos_theta)))

    return angle_degrees

In [29]:
# load the dataframe
df = pd.read_csv("/data/lkolmar/datasets/realistic/config/simulation.csv")
df = df[(df["finished"] == True) & (df["path"] != "not set")]
print(f"Loaded {len(df)} finished simulations")

Loaded 7784 finished simulations


In [30]:
valid_labels = []
threshold = 20  # degrees
count = 0
for i in range(len(df)):
    sample = df.iloc[i]
    vx, vy, vz = sample["rotation_x"], sample["rotation_y"], sample["rotation_z"]
    pos_x = angle_to_axis(vx, vy, vz, (1, 0, 0))  # backspin
    neg_x = angle_to_axis(vx, vy, vz, (-1, 0, 0)) # topspin

    if min(pos_x, neg_x) < threshold:
        valid_labels.append(i)
        if count > 30:
            # break
            pass
        count += 1

print(f"Found {len(valid_labels)} valid labels out of {len(df)}")

Found 384 valid labels out of 7784


In [31]:
print(valid_labels)

[3129, 3130, 3131, 3132, 3139, 3140, 3716, 3717, 3724, 3725, 3726, 3727, 3730, 3731, 3732, 3733, 3734, 3735, 3740, 3741, 3742, 3743, 3744, 3745, 3755, 3756, 3757, 3758, 3772, 3773, 3774, 3775, 4291, 4292, 4306, 4307, 4308, 4309, 4319, 4320, 4321, 4322, 4323, 4324, 4329, 4330, 4331, 4332, 4333, 4334, 4337, 4338, 4339, 4340, 4341, 4342, 4343, 4344, 4349, 4350, 4351, 4352, 4353, 4354, 4355, 4356, 4365, 4366, 4367, 4368, 4369, 4370, 4383, 4384, 4385, 4386, 4387, 4388, 4405, 4406, 4407, 4408, 4427, 4428, 4429, 4430, 4451, 4452, 4859, 4860, 4881, 4882, 4883, 4884, 4898, 4899, 4916, 4917, 4918, 4919, 4920, 4921, 4934, 4935, 4936, 4937, 4938, 4939, 4948, 4949, 4950, 4951, 4952, 4953, 4954, 4955, 4960, 4961, 4962, 4963, 4964, 4965, 4966, 4967, 4969, 4970, 4971, 4972, 4973, 4974, 4975, 4976, 4977, 4978, 4982, 4983, 4984, 4985, 4986, 4987, 4988, 4989, 4998, 4999, 5000, 5001, 5002, 5003, 5004, 5005, 5017, 5018, 5019, 5020, 5021, 5022, 5023, 5024, 5039, 5040, 5041, 5042, 5043, 5044, 5062, 5063, 506

In [13]:
print(valid_labels)

[785, 799, 1108, 1109, 1125, 1126, 1127, 1142, 1143, 1144, 1145, 1159, 1160, 1161, 1173, 1174, 1175, 1185, 1477, 1497, 1498, 1499, 1517, 1518, 1519, 1520, 1521, 1537, 1538, 1539, 1540, 1541]


In [32]:
x, y, z = df.iloc[valid_labels[0]][["rotation_x", "rotation_y", "rotation_z"]]
print(f"Example rotation vector: ({x}, {y}, {z})")
print(f"Angle to backspin axis: {angle_to_axis(x, y, z, (1, 0, 1)):.1f} degrees")
print(f"Angle to topspin axis: {angle_to_axis(x, y, z, (-1, 0, 0)):.1f} degrees")

Example rotation vector: (-130.3448275862069, -43.44827586206896, -14.482758620689651)
Angle to backspin axis: 42.2 degrees
Angle to topspin axis: 19.4 degrees


In [33]:
backspin = 0
topspin = 0
rpss = []
high = 0
mid = 0
low = 0
for i in valid_labels:
    sample = df.iloc[i]
    vx, vy, vz = sample["rotation_x"], sample["rotation_y"], sample["rotation_z"]
    if vx > 0:
        label = "backspin"
        backspin += 1
    else:
        label = "topspin"
        topspin += 1

    path = "/data/lkolmar/datasets/realistic/" + sample["path"]
    metadata = pd.read_csv(path + "metadata.csv")
    rps = metadata["rotation_omega"].values[0]
    rpss.append(rps)
    print(f"Index {i}: {label}, {rps:.1f}")

    if rps >= 100:
        high += 1
    elif rps >= 70:
        mid += 1
    else:
        low += 1

print(f"Backspin: {backspin}, Topspin: {topspin}, Mean RPS: {np.mean(rpss):.1f} +/- {np.std(rpss):.1f}")
print(f"Max rps: {np.max(rpss):.1f}, Min rps: {np.min(rpss):.1f}")
print(f"High (>100 rps): {high}, Mid (70-100 rps): {mid}, Low (<70 rps): {low}")

Index 3129: topspin, 138.2
Index 3130: topspin, 137.5
Index 3131: topspin, 137.5
Index 3132: topspin, 138.2
Index 3139: topspin, 128.4
Index 3140: topspin, 128.4
Index 3716: backspin, 128.4
Index 3717: backspin, 128.4
Index 3724: backspin, 138.2
Index 3725: backspin, 137.5
Index 3726: backspin, 137.5
Index 3727: backspin, 138.2
Index 3730: topspin, 136.8
Index 3731: topspin, 135.4
Index 3732: topspin, 134.7
Index 3733: topspin, 134.7
Index 3734: topspin, 135.4
Index 3735: topspin, 136.8
Index 3740: topspin, 127.6
Index 3741: topspin, 126.2
Index 3742: topspin, 125.4
Index 3743: topspin, 125.4
Index 3744: topspin, 126.2
Index 3745: topspin, 127.6
Index 3755: topspin, 117.0
Index 3756: topspin, 116.2
Index 3757: topspin, 116.2
Index 3758: topspin, 117.0
Index 3772: topspin, 107.8
Index 3773: topspin, 107.0
Index 3774: topspin, 107.0
Index 3775: topspin, 107.8
Index 4291: backspin, 107.0
Index 4292: backspin, 107.8
Index 4306: backspin, 117.0
Index 4307: backspin, 116.2
Index 4308: backsp

FileNotFoundError: [Errno 2] No such file or directory: '/data/lkolmar/datasets/realistic/data/05056/05056_metadata.csv'

In [10]:
label_names = {
    "topspin_slow": 0,
    "topspin_mid": 1,
    "topspin_fast": 2,
    "backspin_slow": 3,
    "backspin_mid": 4,
    "backspin_fast": 5,
}

# Select valid balls and copy to new subset dataset

In [11]:
new_index = 0
labels = []
count = 0
for i in valid_labels:
    sample = df.iloc[i]
    vx, vy, vz = sample["rotation_x"], sample["rotation_y"], sample["rotation_z"]
    path = "/data/lkolmar/datasets/realistic/" + sample["path"]
    metadata = pd.read_csv(path + "metadata.csv")
    rps = metadata["rotation_omega"].values[0]

    if vx > 0:
        label = "backspin"
        if rps >= 100:
            labels.append(label_names["backspin_fast"])
        elif rps >= 70:
            labels.append(label_names["backspin_mid"])
        else:
            labels.append(label_names["backspin_slow"])

    else:
        label = "topspin"
        if rps >= 100:
            labels.append(label_names["topspin_fast"])
        elif rps >= 70:
            labels.append(label_names["topspin_mid"])
        else:
            labels.append(label_names["topspin_slow"])

    index_str = str(new_index).zfill(5)
    os.makedirs(f"/data/lkolmar/datasets/subset/preprocessed/{index_str}", exist_ok=True)
    new_path = f"/data/lkolmar/datasets/subset/preprocessed/{index_str}/{index_str}_roi.hdf5"
    # print(path[-6:-1])
    shutil.copyfile(f"/data/lkolmar/datasets/realistic/preprocessed/{path[-6:-1]}/{path[-6:-1]}_roi.hdf5", new_path)
    new_index += 1
    # print(f"Copied {new_path} with label {label} and {rps:.1f} rps")
    # if new_index >= 10:
    # break
    if count > 30:
        break
    count += 1

df_labels = pd.DataFrame({'index': range(len(labels)), 'label': labels})
print(df_labels)

FileNotFoundError: [Errno 2] No such file or directory: '/data/lkolmar/datasets/realistic/preprocessed/01108/01108_roi.hdf5'

In [29]:
df_labels.to_csv("/data/lkolmar/datasets/subset/config/labels.csv", index=False)

## Relable already preprocessed to top or back

In [6]:
lable_names = {
    0: "topspin",
    1: "backspin"
}

In [7]:
labels = []
indices = []
df_subset = pd.read_csv("/data/lkolmar/datasets/subset/config/labels_6_classes.csv")
for i in range(len(df_subset)):
    label = df_subset.iloc[i]["label"]
    if label in [0, 1, 2]:
        labels.append(0)
    else:
        labels.append(1)
    indices.append(i)

df_subset["label"] = labels
df_subset["index"] = indices
df_subset.to_csv("/data/lkolmar/datasets/subset/config/labels.csv", index=False)

In [8]:
print(len(df_subset[df_subset["label"] == 0]))
print(len(df_subset[df_subset["label"] == 1]))

83
324
